# reproduce-gpt2 — Colab GPU training


In [1]:
!nvidia-smi

Thu Jul 23 15:23:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
 %pip install -q tiktoken

%cd /content/reproduce-gpt2
!git pull

!curl -sL --create-dirs -o data/input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!ls -lh data/input.txt

/content/reproduce-gpt2
Already up to date.
-rw-r--r-- 1 root root 1.1M Jul 23 15:23 data/input.txt


## 2. Imports + device

In [10]:
import torch
import torch.nn.functional as F
import tiktoken
from gpt2.model import GPT, GPTConfig, DataLoaderLite

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

device: cuda
GPU   : Tesla T4


In [11]:
B, T = 4, 1024
train_loader = DataLoaderLite(B=B, T=T)

model = GPT(GPTConfig())
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print('params:', sum(p.numel() for p in model.parameters()))

params: 124439808


In [12]:
import time 
for i in range(50):
    t0 = time.time()
    optimizer.zero_grad()
    xb, yb = train_loader.next_batch()
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 -t0)*1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1-t0)
    print(f"step {i}, loss: {loss.item()}, dt: {dt:2f} ms, tok/sec: {tokens_per_sec}")

step 0, loss: 10.928553581237793, dt: 1266.477346 ms, tok/sec: 3234.1676000580574
step 1, loss: 9.525257110595703, dt: 1190.869093 ms, tok/sec: 3439.5048324610043
step 2, loss: 8.986088752746582, dt: 1195.244312 ms, tok/sec: 3426.9144457711595
step 3, loss: 8.6998929977417, dt: 1189.882517 ms, tok/sec: 3442.35665450879
step 4, loss: 8.393762588500977, dt: 1195.938826 ms, tok/sec: 3424.924345875337
step 5, loss: 8.02257251739502, dt: 1198.580980 ms, tok/sec: 3417.3744347016927
step 6, loss: 7.910907745361328, dt: 1196.088791 ms, tok/sec: 3424.4949297953262
step 7, loss: 7.710430145263672, dt: 1199.464560 ms, tok/sec: 3414.8570438124725
step 8, loss: 7.629522323608398, dt: 1206.770182 ms, tok/sec: 3394.183964986296
step 9, loss: 7.342772006988525, dt: 1205.133677 ms, tok/sec: 3398.7930797830218
step 10, loss: 7.358564376831055, dt: 1198.534966 ms, tok/sec: 3417.5056363412123
step 11, loss: 7.35311222076416, dt: 1214.001417 ms, tok/sec: 3373.966407372035
step 12, loss: 7.409618854522705, 

In [13]:
torch.set_float32_matmul_precision("high") # TF32,  Ampere(A100,compute capability 8.0)

for i in range(50):
    t0 = time.time()
    optimizer.zero_grad()
    xb, yb = train_loader.next_batch()
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 -t0)*1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1-t0)
    print(f"step {i}, loss: {loss.item()}, dt: {dt:.2f} ms, tok/sec: {tokens_per_sec}")

step 0, loss: 6.1824212074279785, dt: 1308.46 ms, tok/sec: 3130.399284558013
step 1, loss: 6.106289386749268, dt: 1294.33 ms, tok/sec: 3164.5631771719422
step 2, loss: 6.506070137023926, dt: 1302.61 ms, tok/sec: 3144.4586244298107
step 3, loss: 6.441607475280762, dt: 1306.44 ms, tok/sec: 3135.236325073559
step 4, loss: 6.22109317779541, dt: 1304.77 ms, tok/sec: 3139.260938525567
step 5, loss: 6.334710597991943, dt: 1302.62 ms, tok/sec: 3144.4379052978875
step 6, loss: 6.574532508850098, dt: 1317.12 ms, tok/sec: 3109.8084220335404
step 7, loss: 6.489353179931641, dt: 1315.02 ms, tok/sec: 3114.780194923631
step 8, loss: 6.181990146636963, dt: 1316.76 ms, tok/sec: 3110.6603547664536
step 9, loss: 6.361390590667725, dt: 1320.78 ms, tok/sec: 3101.2083222768456
step 10, loss: 6.222801208496094, dt: 1320.33 ms, tok/sec: 3102.2650470480676
step 11, loss: 6.177414894104004, dt: 1317.86 ms, tok/sec: 3108.0694063596507
step 12, loss: 6.316008567810059, dt: 1324.60 ms, tok/sec: 3092.262101267495
s

In [14]:
# for T4 GPU, can use fp16 autocast + GradScaler

from torch.cuda.amp import GradScaler
scaler = GradScaler()

for i in range(50):
    t0 = time.time()
    optimizer.zero_grad()
    xb, yb = train_loader.next_batch()
    xb, yb = xb.to(device), yb.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.float16):   # ← fp16 Tensor Core
        logits, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)
    print(f"step {i}, loss: {loss.item():.4f}, dt: {dt:.2f} ms, tok/sec: {tokens_per_sec:.0f}")

/tmp/ipykernel_4372/1849490083.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


step 0, loss: 5.8831, dt: 510.75 ms, tok/sec: 8020
step 1, loss: 5.8077, dt: 504.03 ms, tok/sec: 8127
step 2, loss: 6.0481, dt: 507.69 ms, tok/sec: 8068
step 3, loss: 5.7966, dt: 506.31 ms, tok/sec: 8090
step 4, loss: 5.9497, dt: 512.01 ms, tok/sec: 8000
step 5, loss: 6.0331, dt: 505.83 ms, tok/sec: 8098
step 6, loss: 6.1017, dt: 508.61 ms, tok/sec: 8053
step 7, loss: 6.0878, dt: 504.66 ms, tok/sec: 8116
step 8, loss: 5.9133, dt: 506.70 ms, tok/sec: 8084
step 9, loss: 5.9533, dt: 508.99 ms, tok/sec: 8047
step 10, loss: 5.9283, dt: 506.81 ms, tok/sec: 8082
step 11, loss: 5.8218, dt: 502.87 ms, tok/sec: 8145
step 12, loss: 5.6933, dt: 509.96 ms, tok/sec: 8032
step 13, loss: 5.6283, dt: 506.17 ms, tok/sec: 8092
step 14, loss: 5.7294, dt: 512.07 ms, tok/sec: 7999
step 15, loss: 5.8799, dt: 506.01 ms, tok/sec: 8095
step 16, loss: 5.9146, dt: 504.57 ms, tok/sec: 8118
step 17, loss: 5.8769, dt: 507.42 ms, tok/sec: 8072
step 18, loss: 5.7050, dt: 509.12 ms, tok/sec: 8045
step 19, loss: 5.8938,

Plain fp32 here so it matches the local run and stays easy to debug. Loss should fall past the
~6.3 unigram wall now that the GPU lets us take many more steps.

*(Speed later: on Ampere+ GPUs (A100) wrap the forward in `torch.autocast(device_type='cuda', dtype=torch.bfloat16)`.
On a T4 (Turing) bf16 isn't Tensor-Core accelerated — use fp16 autocast + `torch.cuda.amp.GradScaler` instead.


But also limit to memory bandwidth

In [ ]:
# for T4 GPU, can use fp16 autocast 

from torch.cuda.amp import GradScaler
scaler = GradScaler()

for i in range(50):
    t0 = time.time()
    optimizer.zero_grad()
    xb, yb = train_loader.next_batch()
    xb, yb = xb.to(device), yb.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):   # ← bf16 
        logits, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)
    print(f"step {i}, loss: {loss.item():.4f}, dt: {dt:.2f} ms, tok/sec: {tokens_per_sec:.0f}")